In [611]:
import pypowsybl.network as pn
import pypowsybl as pp
from pypowsybl_jupyter import nad_explorer
import pandas as pd 
import importlib

In [612]:
from utils import extract_slack_info_for_nad, unique_list
from nad_explorer_slack import nad_explorer_with_slack
import nad_explorer_slack
import pypowsybl as pp
import utils
import importlib


In [613]:
importlib.reload(utils)
importlib.reload(nad_explorer_slack)

<module 'nad_explorer_slack' from 'd:\\dev\\powsybl-open-loadflow-knitro-solver\\nad_explorer_slack.py'>

In [614]:
data = pd.read_csv(
   "test_active_power_IEEE\ieee300\ieee300_active-perturbation.csv",
    sep=";",
    engine="python",
    index_col=False,
   # on_bad_lines="skip",
)
data.head()
len(data)
data["has_slack"] = data['slackValue'].notna()


In [615]:
data.head()

,busId,type,slackValue_pu,slackValue,gen,controlevoltage,transfo,shunt,load,load_violation,gen_violation,has_slack
0,VL1_0,P,-0.087044,-8.704352,NaN,NaN,NaN,NaN,VL1_0_load,0,0,True
1,VL3_0,P,-0.013575,-1.357488,NaN,NaN,NaN,NaN,VL3_0_load,1,0,True
2,VL4_0,P,-0.009528,-0.952793,NaN,NaN,NaN,NaN,NaN,0,0,True
3,VL5_0,P,-0.060539,-6.053853,NaN,NaN,NaN,NaN,VL5_0_load,1,0,True
4,VL5_1,P,-0.000181,-0.018121,NaN,NaN,NaN,NaN,VL5_1_load,0,0,True


In [616]:
network = pp.network.load('Outputs\ieee300-active-perturbation.xiidm')
network.get_bus_breaker_view_buses()

,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id,bus_id
id,,,,,,,
B1,1,113.522755,-113.852793,0,0,VL1,VL1_0
B2,1,120.215472,-85.929935,0,0,VL1,VL1_1
B3,1,239.794933,-87.544100,0,0,VL3,VL3_0
B4,1,371.752170,-86.144252,0,0,VL4,VL4_0
B7001,1,14.499660,-108.715315,0,0,VL7001,VL7001_0
...,...,...,...,...,...,...,...
B9023,1,6.436462,-68.074921,0,0,VL9023,VL9023_0
B9025,1,0.579254,-69.090107,0,0,VL9025,VL9025_0
B9026,1,0.579756,-69.004659,0,0,VL9025,VL9025_1


In [617]:
# data_grouped = data.groupby('busId')[['type','slackValue_pu','slackValue']].agg(unique_list)
# len(data_grouped)

In [618]:
# network = pp.network.load('Outputs/ieee300-1.xiidm')
slack_info = network.get_bus_breaker_view_buses().join(
    data.set_index("busId"),
    on="bus_id",
    how="left"
)
slack_info.reset_index(inplace=True)
slack_info

,id,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id,bus_id,type,slackValue_pu,slackValue,gen,controlevoltage,transfo,shunt,load,load_violation,gen_violation,has_slack
0,B1,1,113.522755,-113.852793,0,0,VL1,VL1_0,P,-0.087044,-8.704352,NaN,NaN,NaN,NaN,VL1_0_load,0.0,0.0,True
1,B2,1,120.215472,-85.929935,0,0,VL1,VL1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,B3,1,239.794933,-87.544100,0,0,VL3,VL3_0,P,-0.013575,-1.357488,NaN,NaN,NaN,NaN,VL3_0_load,1.0,0.0,True
3,B4,1,371.752170,-86.144252,0,0,VL4,VL4_0,P,-0.009528,-0.952793,NaN,NaN,NaN,NaN,NaN,0.0,0.0,True
4,B7001,1,14.499660,-108.715315,0,0,VL7001,VL7001_0,P,-0.085616,-8.561607,B7001-G,"VoltageControl(type=GENERATOR, controlledBus='...",NaN,NaN,NaN,0.0,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,B9023,1,6.436462,-68.074921,0,0,VL9023,VL9023_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
299,B9025,1,0.579254,-69.090107,0,0,VL9025,VL9025_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
300,B9026,1,0.579756,-69.004659,0,0,VL9025,VL9025_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
301,B9044,1,6.515837,-68.229800,0,0,VL9044,VL9044_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [619]:
# Aggregate bus-level slack data to voltage level
slack_info_by_vl = slack_info.groupby('voltage_level_id').agg({
    'has_slack': 'any',  # Has slack if ANY bus in VL has slack
    'type': lambda x: list(x.dropna()),  # List of slack types in VL,
    'slackValue_pu': lambda x: list(x.dropna()),  # List of slack values in VL,
    'bus_id': lambda x: list(x.dropna().unique())  # List of bus IDs in VL
}).reset_index()

# Rename for compatibility
slack_info_by_vl.set_index('voltage_level_id', inplace=True)

slack_info_by_vl

,has_slack,type,slackValue_pu,bus_id
voltage_level_id,,,,
VL1,True,[P],[-0.087044],"[VL1_0, VL1_1]"
VL10,True,"[P, P]","[-0.018232, -0.017848]","[VL10_0, VL10_1]"
VL100,False,[],[],[VL100_0]
VL102,True,[P],[-0.018599],[VL102_0]
VL103,False,[],[],[VL103_0]
...,...,...,...,...
VL94,True,[P],[-0.044341],[VL94_0]
VL9533,False,[],[],[VL9533_0]
VL97,False,[],[],[VL97_0]


In [620]:
explorer = nad_explorer_with_slack(network, slack_info=slack_info_by_vl)

In [621]:
explorer

In [622]:
import pypowsybl.network as pn
from pypowsybl_jupyter import network_explorer, display_nad